# Lab Assignment 6

Efficient Fine-Tuning and Alignment of LLMs: LoRA, QLoRA and RLHF

## Objective
A pretrained language model has billions of parameters. Adapting it to a new task, or aligning it with human
preferences, with naive *full fine-tuning* is far too expensive for most labs. This assignment walks through the
three ideas that make adaptation practical, from the cheapest trick to the most involved procedure:

1. **LoRA (Part A)** — freeze the pretrained weights and learn only a tiny *low-rank* update. You implement the
   forward pass, the gradients, and training by hand, and discover how the rank controls what can be learned.
2. **QLoRA (Part B)** — additionally store the frozen base weights in **4-bit**, so that big models fit on small
   GPUs. You implement block-wise quantisation (uniform int4 and the NormalFloat NF4 code book) and train
   LoRA adapters on top of a quantised base.
3. **RLHF (Part C)** — align a model with human preferences. You train a **reward model** from pairwise
   comparisons (Bradley–Terry), then optimise a policy against it with a **KL penalty** to the reference model,
   and watch **reward hacking** happen when the KL penalty is too weak. (Bonus: DPO.)
4. **The real libraries (Part D)** — repeat LoRA and QLoRA on a real transformer using `peft` and `bitsandbytes`.

Parts A–C use a *toy* "LLM layer" and a *toy* response space so that everything runs in seconds on a laptop CPU
and every gradient can be derived and verified by hand. The mathematics is exactly what real systems use.

> **Read Tutorial 6 first** (`Tutorial_6_LoRA_QLoRA_RLHF.ipynb`). It explains each idea with small worked examples
> and tells you what each task below expects.

## Submission
- Complete all `TODO` sections and all markdown answers.
- Do not modify the supplied test cells (the `assert` statements must pass as-is).
- Parts A, B and C: only `numpy`, `matplotlib`, `random` and the Python standard library — no ML libraries.
  Part D is the one place you are allowed (in fact required) to use `torch`, `transformers`, `peft` and
  `bitsandbytes`. Part D is meant for Google Colab (free T4 GPU). If you have no GPU, do D1 and answer D2 in
  words instead (see the note in Part D).
- Run **Kernel > Restart & Run All** before submitting the completed `.ipynb` file. Cells run in order and later
  cells reuse functions defined earlier (for example `numeric_grad` from Task A2).
- Keep the random seeds as given so that your numbers are reproducible.

## Structure
| Part | Content |
|---|---|
| A | LoRA: init/forward/merge, backward + gradient check, training + rank sweep |
| B | QLoRA: block-wise int4, NF4, QLoRA training + memory |
| C | RLHF: reward model, KL-regularised policy optimisation, reward hacking |
| D | `peft` LoRA and `bitsandbytes` QLoRA on a real model |
| Analysis | Five short questions at the end |
| Bonus | DPO (Task C4) |

# Part A: LoRA from Scratch

A linear layer computes $y = xW_0$ with a pretrained weight $W_0 \in \mathbb{R}^{d_{in}\times d_{out}}$
(we use the *row-vector* convention: $x$ is a row, a batch is a matrix with one row per example).

**Full fine-tuning** would update all $d_{in} d_{out}$ entries of $W_0$. **LoRA** (Low-Rank Adaptation) instead
keeps $W_0$ **frozen** and learns a low-rank correction:

$$
y = x W_0 + \frac{\alpha}{r}\, x A B, \qquad A \in \mathbb{R}^{d_{in}\times r},\; B \in \mathbb{R}^{r\times d_{out}},\; r \ll \min(d_{in}, d_{out}).
$$

Only $A$ and $B$ are trained: $r(d_{in}+d_{out})$ numbers instead of $d_{in}d_{out}$. The factor $\alpha/r$ is a
fixed scale. Initialisation matters: $A$ is random and **$B$ is zero**, so at the start the adapted model is
*exactly* the pretrained model.

### Task A1: Initialisation, forward pass, merging

1. `lora_init(d_in, d_out, r)` — return `A` (shape `d_in x r`, entries $\mathcal{N}(0, 1/d_{in})$, i.e.
   `np.random.randn(d_in, r) / np.sqrt(d_in)`) and `B` (zeros, shape `r x d_out`).
2. `lora_forward(x, W0, A, B, alpha)` — the formula above, with `r = A.shape[1]`.
3. `merge_lora(W0, A, B, alpha)` — return a single matrix $W = W_0 + \frac{\alpha}{r}AB$. After training, merging
   removes all inference overhead.
4. `count_params(*arrays)` — total number of scalar entries in the given arrays.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

random.seed(0)
np.random.seed(0)


def lora_init(d_in, d_out, r):
    # TODO: A ~ N(0, 1/d_in) with shape (d_in, r); B = zeros with shape (r, d_out)
    pass


def lora_forward(x, W0, A, B, alpha):
    # TODO: x @ W0 + (alpha / r) * (x @ A) @ B     (r = A.shape[1])
    pass


def merge_lora(W0, A, B, alpha):
    # TODO: W0 + (alpha / r) * A @ B
    pass


def count_params(*arrays):
    # TODO: total number of entries over all arrays
    pass


# --- test cases: do not modify ---
np.random.seed(0)
d_in, d_out, r = 16, 12, 4
W0 = np.random.randn(d_in, d_out)
A, B = lora_init(d_in, d_out, r)
assert A.shape == (d_in, r) and B.shape == (r, d_out)
assert np.allclose(B, 0) and not np.allclose(A, 0)

x = np.random.randn(5, d_in)
# with B = 0 the adapted layer must equal the base layer
assert np.allclose(lora_forward(x, W0, A, B, alpha=8), x @ W0)

# with a non-zero B, merging must give identical outputs
B_test = np.random.randn(r, d_out)
y_lora = lora_forward(x, W0, A, B_test, alpha=8)
W_merged = merge_lora(W0, A, B_test, alpha=8)
assert W_merged.shape == W0.shape
assert np.allclose(y_lora, x @ W_merged)

assert count_params(A, B) == r * (d_in + d_out)
assert count_params(W0) == d_in * d_out
print("Task A1 checks passed.")
print(f"Full layer: {count_params(W0)} params | LoRA r={r}: {count_params(A, B)} params")

### Task A2: Backward pass and gradient check

We use the loss $L = \frac{1}{2N}\sum_{n=1}^{N}\lVert \hat y_n - y_n\rVert^2$ (`mse_loss`, supplied).
With $\hat Y = XW_0 + s\,XAB$ and $s=\alpha/r$, let $dY = \partial L/\partial \hat Y = (\hat Y - Y)/N$. Then

$$
\frac{\partial L}{\partial B} = s\,(XA)^\top dY, \qquad
\frac{\partial L}{\partial A} = s\,X^\top\!\big(dY\,B^\top\big).
$$

$W_0$ is **frozen**: it receives no gradient and is never updated. Implement `lora_backward` returning `dA, dB`.

You then *prove* your gradients are right with a **numerical gradient check**: compare against central finite
differences $\frac{f(\theta+\epsilon)-f(\theta-\epsilon)}{2\epsilon}$. `numeric_grad` is supplied. Gradient
checking is the single most useful debugging tool when writing backprop by hand.

In [ ]:
def mse_loss(y_pred, y_true):
    # supplied: L = 1/(2N) * sum_n ||y_pred_n - y_true_n||^2
    return 0.5 * np.mean(np.sum((y_pred - y_true) ** 2, axis=1))


def numeric_grad(f, X, eps=1e-6):
    # supplied: central-difference gradient of scalar function f at array X
    g = np.zeros_like(X, dtype=float)
    it = np.nditer(X, flags=["multi_index"])
    for _ in it:
        i = it.multi_index
        old = X[i]
        X[i] = old + eps; fp = f(X)
        X[i] = old - eps; fm = f(X)
        X[i] = old
        g[i] = (fp - fm) / (2 * eps)
    return g


def lora_backward(x, y_pred, y_true, A, B, alpha):
    # TODO: return dA, dB  (W0 is frozen, so it gets no gradient)
    #   N = len(x); s = alpha / A.shape[1]; dY = (y_pred - y_true) / N
    #   dB = s * (x @ A).T @ dY
    #   dA = s * x.T @ (dY @ B.T)
    pass


# --- test cases: do not modify ---
np.random.seed(3)
d_in_t, d_out_t, r_t, N_t = 6, 5, 3, 8
W0_t = np.random.randn(d_in_t, d_out_t)
A_t = np.random.randn(d_in_t, r_t)
B_t = np.random.randn(r_t, d_out_t)          # non-zero, otherwise dA would be exactly 0
x_t = np.random.randn(N_t, d_in_t)
y_t = np.random.randn(N_t, d_out_t)
alpha_t = 6

y_pred_t = lora_forward(x_t, W0_t, A_t, B_t, alpha_t)
dA_t, dB_t = lora_backward(x_t, y_pred_t, y_t, A_t, B_t, alpha_t)
assert dA_t.shape == A_t.shape and dB_t.shape == B_t.shape

num_dA = numeric_grad(lambda A_: mse_loss(lora_forward(x_t, W0_t, A_, B_t, alpha_t), y_t), A_t)
num_dB = numeric_grad(lambda B_: mse_loss(lora_forward(x_t, W0_t, A_t, B_, alpha_t), y_t), B_t)
rel_err = lambda a, b: np.linalg.norm(a - b) / (np.linalg.norm(a) + np.linalg.norm(b) + 1e-12)
print("relative error dA:", rel_err(dA_t, num_dA), "| dB:", rel_err(dB_t, num_dB))
assert rel_err(dA_t, num_dA) < 1e-5
assert rel_err(dB_t, num_dB) < 1e-5
print("Task A2 gradient check passed.")

### Task A3: Train LoRA and sweep the rank

**The toy task.** A "pretrained" layer `W_base` (64 x 64) has to be adapted to a new task whose true weight is
`W_target = W_base + Delta`, where `Delta` is a **rank-2** matrix (that is the whole premise of LoRA: the
change needed for a new task is low-rank). We observe noisy input/output pairs from `W_target`.

1. Write `train_lora(x, y, W0, r, alpha, lr, steps, seed)`: initialise with `lora_init`, then for each step do
   forward &rarr; loss (append to `losses`) &rarr; `lora_backward` &rarr; gradient-descent update of **`A` and `B`
   only**. Return `A, B, losses`.
2. Plot the training-loss curve (log-scale y-axis) for `r = 2`.
3. **Rank sweep:** train with `r` in `[1, 2, 4, 8, 16]` (use `alpha = 2 * r`), record the **validation** loss and
   the number of trainable parameters for each, and plot validation loss against `r`, with the full fine-tuning
   result (supplied function `full_finetune`) as a horizontal reference line.

In [ ]:
# --- supplied: the toy fine-tuning task (do not modify) ---
np.random.seed(1)
D = 64
W_base = np.random.randn(D, D) / np.sqrt(D)                     # "pretrained" weight
Delta = 0.03 * (np.random.randn(D, 2) @ np.random.randn(2, D))  # the needed change: rank 2
W_target = W_base + Delta

X_train = np.random.randn(1024, D)
X_val = np.random.randn(256, D)
Y_train = X_train @ W_target + 0.01 * np.random.randn(1024, D)
Y_val = X_val @ W_target + 0.01 * np.random.randn(256, D)

print("Validation loss of the un-adapted base model:", mse_loss(X_val @ W_base, Y_val))
print("Rank of the true update Delta:", np.linalg.matrix_rank(Delta))


def full_finetune(x, y, W0, lr=0.1, steps=500):
    # supplied baseline: update ALL entries of W (D*D parameters)
    W = W0.copy()
    N = len(x)
    for _ in range(steps):
        dY = (x @ W - y) / N
        W -= lr * (x.T @ dY)
    return W

In [ ]:
def train_lora(x, y, W0, r, alpha, lr=0.1, steps=500, seed=0):
    np.random.seed(seed)
    A, B = lora_init(W0.shape[0], W0.shape[1], r)
    losses = []
    # TODO: for each step
    #   y_pred = lora_forward(...); losses.append(mse_loss(...))
    #   dA, dB = lora_backward(...)
    #   A -= lr * dA ; B -= lr * dB          (W0 is never touched)
    return A, B, losses


# --- test cases: do not modify ---
W_base_copy = W_base.copy()
A2_, B2_, losses = train_lora(X_train, Y_train, W_base, r=2, alpha=4)
assert np.array_equal(W_base, W_base_copy), "the frozen weight must not change"
val_loss_r2 = mse_loss(lora_forward(X_val, W_base, A2_, B2_, 4), Y_val)
print(f"train loss: {losses[0]:.4f} -> {losses[-1]:.5f} | validation loss (r=2): {val_loss_r2:.5f}")
assert len(losses) == 500
assert losses[-1] < 0.01 * losses[0]
assert val_loss_r2 < 0.05
assert count_params(A2_, B2_) == 2 * (D + D)
print("Task A3 (training) checks passed.")

In [ ]:
# TODO: plot the r = 2 training-loss curve (log-scale y axis), with axis labels and a title
# plt.figure(figsize=(5, 3.5))
# plt.semilogy(losses)
# plt.xlabel("Step"); plt.ylabel("Training loss"); plt.title("LoRA (r=2) training loss")
# plt.show()

In [ ]:
ranks = [1, 2, 4, 8, 16]
sweep_val_loss = {}      # r -> validation loss
sweep_params = {}        # r -> number of trainable parameters

# TODO: for each r in ranks: train_lora(..., r=r, alpha=2*r), compute the validation loss with lora_forward
#       and the parameter count with count_params, store them in the dictionaries above.

# TODO: W_full = full_finetune(X_train, Y_train, W_base); full_val = mse_loss(X_val @ W_full, Y_val)
full_val = None

# TODO: plot validation loss (log y) against r, with a horizontal line for full fine-tuning,
#       and print a small table: r | trainable params | validation loss

# --- test cases: do not modify ---
assert set(sweep_val_loss) == set(ranks) and set(sweep_params) == set(ranks)
assert sweep_val_loss[1] > 10 * sweep_val_loss[2], "rank 1 cannot represent a rank-2 update"
assert sweep_val_loss[2] < 0.05
assert sweep_params[16] == 16 * (D + D)
print("Task A3 (rank sweep) checks passed.")

# Part B: QLoRA — a 4-bit Frozen Base

LoRA shrinks the *trainable* parameters, but the frozen base model still has to sit in GPU memory. A 7B-parameter
model in 16-bit needs about 14 GB just for the weights. **QLoRA** stores the frozen base weights in **4 bits**
and *de-quantises them on the fly* inside the forward pass, while the LoRA adapters $A, B$ stay in higher
precision and receive all the gradients:

$$
y = x\,\underbrace{\mathrm{deq}(W_0^{4\text{-bit}})}_{\text{frozen}} + \frac{\alpha}{r}\, x A B .
$$

### Task B1: Block-wise 4-bit quantisation

Quantising a whole tensor with one scale is fragile: a single outlier stretches the scale and crushes all the
other weights to zero. So weights are cut into **blocks** (we use 64 values), each with its own scale.

For each block: `absmax = max(|w|)`, then `code = round(w / absmax * 7)` which is an integer in $[-7, 7]$ (that
is 15 levels, a symmetric signed 4-bit code), and de-quantisation is `code / 7 * absmax`.

1. `quantize_blockwise(w, block_size=64)` — flatten `w`, reshape to `(-1, block_size)`, return
   `codes` (dtype `np.int8`, shape `(n_blocks, block_size)`) and `absmax` (shape `(n_blocks, 1)`). Handle blocks
   whose `absmax` is 0 (no division by zero).
2. `dequantize_blockwise(codes, absmax, shape)` — return the reconstruction reshaped to `shape`.
3. **Block-size experiment:** for a tensor with one big outlier, compare the reconstruction MSE using
   `block_size = 64` against a single block for the whole tensor (`block_size = 4096`).

In [ ]:
def quantize_blockwise(w, block_size=64):
    # TODO: blocks = w.reshape(-1, block_size); absmax = max |.| per block (keep dims);
    #       codes = np.round(blocks / safe_absmax * 7).astype(np.int8)   (avoid dividing by 0)
    # return codes, absmax
    pass


def dequantize_blockwise(codes, absmax, shape):
    # TODO: codes / 7 * absmax, reshaped to `shape`
    pass


# --- test cases: do not modify ---
np.random.seed(5)
w_test = np.random.randn(8, 64) * 0.1
codes, absmax = quantize_blockwise(w_test, 64)
assert codes.dtype == np.int8 and codes.min() >= -7 and codes.max() <= 7
assert codes.shape == (8, 64) and absmax.shape == (8, 1)
w_hat = dequantize_blockwise(codes, absmax, w_test.shape)
assert w_hat.shape == w_test.shape
# every reconstruction error is at most half a quantisation step of its own block
assert np.all(np.abs(w_test - w_hat) <= (absmax / 7 / 2) + 1e-12)
# an all-zero block must survive
z = np.zeros((1, 64)); cz, az = quantize_blockwise(z, 64)
assert np.all(dequantize_blockwise(cz, az, z.shape) == 0)
print("Task B1 checks passed.")

In [ ]:
# Block-size experiment: one outlier in a tensor of 4096 weights.
np.random.seed(6)
w_out = np.random.randn(4096) * 0.05
w_out[100] = 5.0                       # a single outlier
mse_by_block = {}                      # block_size -> reconstruction MSE

# TODO: for block_size in [64, 4096]: quantise, de-quantise, store np.mean((w_out - w_hat) ** 2)

# --- test cases: do not modify ---
assert set(mse_by_block) == {64, 4096}
print(mse_by_block)
assert mse_by_block[64] < 0.1 * mse_by_block[4096]
print("Task B1 (block-size experiment) passed.")

### Task B2: NormalFloat-4 (NF4)

Uniform levels are not the best use of only 16 values. Trained weights are roughly **bell-shaped**, so most of
them are small. **NF4** places its 16 levels at (normalised) quantiles of a standard normal distribution: more
levels near 0 where the weights are dense, fewer in the tails. After dividing a block by its `absmax` the values
lie in $[-1,1]$, and each value is snapped to the **nearest** of the 16 NF4 levels (supplied below).

1. `quantize_nf4(w, block_size=64)` — returns `idx` (dtype `np.uint8`, values 0..15, shape `(n_blocks, block_size)`,
   the index of the nearest level) and `absmax` (shape `(n_blocks, 1)`).
2. `dequantize_nf4(idx, absmax, shape)` — `NF4_LEVELS[idx] * absmax`, reshaped.
3. On 32,000 Gaussian weights, compare the reconstruction MSE of NF4 against the uniform int4 of Task B1 and print both.

In [ ]:
NF4_LEVELS = np.array([
    -1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
    -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
    0.07958029955625534, 0.16093020141124725, 0.24611230194568542, 0.33791524171829224,
    0.44070982933044434, 0.5626170039558411, 0.7229568362236023, 1.0,
])  # supplied: the 16 NF4 code-book values


def quantize_nf4(w, block_size=64):
    # TODO: blocks = w.reshape(-1, block_size); absmax per block; normalised = blocks / safe_absmax
    #       idx = index of the nearest NF4 level:  np.abs(normalised[..., None] - NF4_LEVELS).argmin(-1)
    # return idx.astype(np.uint8), absmax
    pass


def dequantize_nf4(idx, absmax, shape):
    # TODO: NF4_LEVELS[idx] * absmax, reshaped to `shape`
    pass


# --- test cases: do not modify ---
np.random.seed(7)
w_gauss = np.random.randn(64 * 500)
idx, am = quantize_nf4(w_gauss, 64)
assert idx.dtype == np.uint8 and idx.max() <= 15 and idx.shape == (500, 64)
w_nf4 = dequantize_nf4(idx, am, w_gauss.shape)
c_u, a_u = quantize_blockwise(w_gauss, 64)
w_int4 = dequantize_blockwise(c_u, a_u, w_gauss.shape)
mse_nf4 = np.mean((w_gauss - w_nf4) ** 2)
mse_int4 = np.mean((w_gauss - w_int4) ** 2)
print(f"MSE int4: {mse_int4:.5f} | MSE NF4: {mse_nf4:.5f}")
assert mse_nf4 < mse_int4
print("Task B2 checks passed.")

### Task B3: Train a QLoRA layer and count the memory

**Part 1 — QLoRA training.** Quantise the frozen base `W_base` from Task A3 to int4 and to NF4 (block size 64),
de-quantise each back to a float matrix (that is what the forward pass would use), and train LoRA (`r = 2`,
`alpha = 4`) on top of it with your `train_lora` from Task A3 — pass the **de-quantised** matrix as `W0`. For each
of `fp` (original `W_base`), `int4` and `nf4` record `(validation loss before training, validation loss after
training)` in the dictionary `qlora_results`. The adapters have to fit the new task *and* can partly repair the
damage done by quantisation.

**Part 2 — memory.** Write `model_memory_gb(n_params, mode, trainable_fraction=0.01, block_size=64)` using this
simplified accounting (activations are ignored):

| mode | bytes per parameter |
|---|---|
| `"full"` | 16 (fp16 weight 2 + fp16 gradient 2 + fp32 master copy 4 + Adam $m,v$ 8) |
| `"lora"` | 2 (frozen fp16 base) $+\ 16\times$ `trainable_fraction` |
| `"qlora"` | $0.5 + 2/$`block_size` (4-bit code + one fp16 `absmax` per block) $+\ 16\times$ `trainable_fraction` |

Return gigabytes (`1 GB = 1e9` bytes).

In [ ]:
# Part 1: QLoRA training on a quantised base
qlora_results = {}     # name -> (val loss before, val loss after)

# TODO: build W_int4 and W_nf4 (quantise + de-quantise W_base, block size 64), then for
#       name, W0 in [("fp", W_base), ("int4", W_int4), ("nf4", W_nf4)]:
#           before = mse_loss(X_val @ W0, Y_val)
#           A_, B_, _ = train_lora(X_train, Y_train, W0, r=2, alpha=4)
#           after = mse_loss(lora_forward(X_val, W0, A_, B_, 4), Y_val)
#           qlora_results[name] = (before, after)

for k, (b, a) in qlora_results.items():
    print(f"{k:>5}: before {b:.4f} -> after {a:.4f}")

# --- test cases: do not modify ---
assert set(qlora_results) == {"fp", "int4", "nf4"}
for k, (b, a) in qlora_results.items():
    assert a < 0.1 * b, k
assert qlora_results["fp"][1] < qlora_results["nf4"][1] < qlora_results["int4"][1]
print("Task B3 (training) checks passed.")

In [ ]:
def model_memory_gb(n_params, mode, trainable_fraction=0.01, block_size=64):
    # TODO: bytes per parameter from the table above, times n_params, divided by 1e9
    pass


# --- test cases: do not modify ---
n7b = 7e9
full, lora, qlora = (model_memory_gb(n7b, m) for m in ["full", "lora", "qlora"])
print(f"7B model  full: {full:.1f} GB | LoRA: {lora:.1f} GB | QLoRA: {qlora:.1f} GB")
assert np.isclose(full, 112.0)
assert np.isclose(lora, 15.12)
assert np.isclose(qlora, 4.83875)
assert qlora < lora < full
print("Task B3 (memory) checks passed.")

# Part C: RLHF — Reward Model, KL-Regularised Policy, Reward Hacking

**Reinforcement Learning from Human Feedback** aligns a model with what people prefer, in three stages:

1. **SFT** — supervised fine-tuning gives a reference policy $\pi_{ref}$ (we just take it as given).
2. **Reward model** — humans compare two responses; we fit a scalar reward $r_\phi(x)$ so that the preferred one
   scores higher (**Bradley–Terry**): $P(a \succ b) = \sigma\big(r(a) - r(b)\big)$.
3. **Policy optimisation** — maximise reward while staying close to $\pi_{ref}$:
$$
\max_\pi\; \mathbb{E}_{a\sim\pi}\big[r(a)\big] \;-\; \beta\, \mathrm{KL}\big(\pi \,\Vert\, \pi_{ref}\big).
$$
   The KL term is what stops the policy from "exploiting" the (imperfect) reward model.

**Toy world.** A "response" is described by 3 numbers: how **helpful** it is, how **polite**, and how **long**.
Human raters like helpful and polite answers, mildly like longer answers, but *hate extreme rambling*:

$$
r_{true}(h,p,\ell) = h + 0.5\,p + 0.3\,\ell - 1.0\cdot\max(0,\ell-1.5)^2 .
$$

Our **reward model is linear**, $r_w(x) = w^\top x$, and it will only ever see responses of moderate length
(`length` is clipped to $[-1.5, 1.5]$ in the comparison data), so it cannot know about the rambling penalty.
Keep this in mind for Task C3.

### Task C1: Reward model from pairwise preferences

For a chosen response $x_c$ and a rejected one $x_r$, with $d = x_c - x_r$, the Bradley–Terry loss is
$$
L(w) = -\frac{1}{M}\sum_m \log \sigma(w^\top d_m),\qquad
\nabla_w L = -\frac{1}{M}\sum_m \big(1-\sigma(w^\top d_m)\big)\, d_m .
$$

1. `sigmoid(z)` — **numerically stable** (no overflow warnings for $z=\pm 1000$). Hint: use $e^{-|z|}$.
2. `preference_loss(w, X_chosen, X_rejected)` and `preference_grad(...)`.
   (Hint: $-\log\sigma(z) = \log(1+e^{-z})$ = `np.logaddexp(0, -z)`.)
3. `train_reward_model(X_chosen, X_rejected, lr=0.5, steps=3000)` — gradient descent from `w = 0`; return `w`
   and the list of losses.

In [ ]:
# --- supplied: synthetic preference data (do not modify) ---
def true_reward(X):
    h, p, l = X[:, 0], X[:, 1], X[:, 2]
    return h + 0.5 * p + 0.3 * l - 1.0 * np.maximum(0.0, l - 1.5) ** 2


def sample_responses(rng, n):
    h = rng.randn(n); p = rng.randn(n)
    l = np.clip(0.8 * h + 0.6 * rng.randn(n), -1.5, 1.5)   # moderate lengths only, correlated with helpfulness
    return np.stack([h, p, l], axis=1)


rng_data = np.random.RandomState(0)
pool = sample_responses(rng_data, 400)
r_pool = true_reward(pool)
ia = rng_data.randint(0, 400, 300); ib = rng_data.randint(0, 400, 300)
# humans are noisy: they pick a over b with probability sigmoid(r(a) - r(b))
a_wins = rng_data.rand(300) < 1 / (1 + np.exp(-(r_pool[ia] - r_pool[ib])))
X_chosen = np.where(a_wins[:, None], pool[ia], pool[ib])
X_rejected = np.where(a_wins[:, None], pool[ib], pool[ia])

# a clean held-out set (labels from the true reward, no noise) to measure how well the model ranks
rng_val = np.random.RandomState(1)
Xv_a, Xv_b = sample_responses(rng_val, 1000), sample_responses(rng_val, 1000)
val_label = true_reward(Xv_a) > true_reward(Xv_b)
print("training comparisons:", len(X_chosen), "| held-out pairs:", len(val_label))

In [ ]:
def sigmoid(z):
    # TODO: numerically stable sigmoid. e = np.exp(-np.abs(z));  z>=0 -> 1/(1+e);  z<0 -> e/(1+e)
    pass


def preference_loss(w, X_chosen, X_rejected):
    # TODO: mean of -log sigmoid(w . (x_c - x_r))   (use np.logaddexp(0, -z))
    pass


def preference_grad(w, X_chosen, X_rejected):
    # TODO: -mean( (1 - sigmoid(z))[:, None] * d, axis=0 ) with d = X_chosen - X_rejected, z = d @ w
    pass


def train_reward_model(X_chosen, X_rejected, lr=0.5, steps=3000):
    w = np.zeros(X_chosen.shape[1])
    losses = []
    # TODO: gradient descent; append the loss at every step
    return w, losses


# --- test cases: do not modify ---
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("error")                    # any overflow warning becomes an error
    s = sigmoid(np.array([-1000.0, 0.0, 1000.0]))
assert np.allclose(s, [0.0, 0.5, 1.0])

w_chk = np.array([0.3, -0.2, 0.5])
g = preference_grad(w_chk, X_chosen, X_rejected)
g_num = numeric_grad(lambda w_: preference_loss(w_, X_chosen, X_rejected), w_chk)
assert np.allclose(g, g_num, atol=1e-6), (g, g_num)

w_rm, rm_losses = train_reward_model(X_chosen, X_rejected)
assert rm_losses[-1] < rm_losses[0]
acc = np.mean(((Xv_a @ w_rm) > (Xv_b @ w_rm)) == val_label)
print("learned reward weights [helpful, polite, length]:", np.round(w_rm, 2))
print("true weights inside the training range:           [1.0, 0.5, 0.3]")
print(f"held-out ranking accuracy: {acc:.3f}")
assert acc > 0.9
print("Task C1 checks passed.")

### Task C2: KL-regularised policy optimisation

Now the "language model" is a **policy over 8 candidate responses**: $\pi_\theta(a) = \mathrm{softmax}(\theta)_a$.
The candidates' features, the reference policy $\pi_{ref}$ and the **proxy reward** (your learned reward model
applied to the candidates) are supplied. Implement:

1. `log_softmax(z)` — stable ($z - \max z - \log\sum e^{z-\max z}$).
2. `kl_divergence(p, q)` — $\sum_a p_a(\log p_a - \log q_a)$, with `0 log 0 = 0`.
3. `rlhf_objective(pi, reward, pi_ref, beta)` — $\mathbb{E}_\pi[r]-\beta\,\mathrm{KL}(\pi\Vert\pi_{ref})$.
4. `optimal_policy(reward, pi_ref, beta)` — the **closed-form maximiser**
   $\pi^*(a) \propto \pi_{ref}(a)\,\exp\big(r(a)/\beta\big)$. Compute it stably as
   `softmax(log(pi_ref) + reward / beta)`.
5. `reinforce_step(theta, reward, logp_ref, beta, lr, n_samples, rng)` — one **policy-gradient (REINFORCE)** step.
   Sample `n_samples` actions $a_i \sim \pi_\theta$; use the *shaped* reward
   $\tilde r_i = r(a_i) - \beta\big(\log\pi_\theta(a_i) - \log\pi_{ref}(a_i)\big)$, subtract the batch mean as a
   baseline to get advantages $A_i$, and ascend with
   $\nabla_\theta \approx \frac{1}{n}\sum_i A_i\,(\mathbf{e}_{a_i} - \pi_\theta)$, that is
   `theta += lr * grad`. Return the updated `theta`.
   (Why $\mathbf{e}_a - \pi$? It is $\nabla_\theta \log\pi_\theta(a)$ for a softmax.)

Then train for 400 steps with $\beta = 0.5$ and check that sampling-based RL ends up where the closed form says.

In [ ]:
# --- supplied: the toy response space (do not modify) ---
cands = np.array([[0.2, 0.1, 0.1], [0.8, 0.4, 0.6], [1.2, 0.6, 0.9], [0.0, -0.5, -0.3],
                  [-0.6, 0.2, -0.5], [0.5, 0.9, 0.4], [1.0, 0.5, 6.0], [1.0, 0.3, 1.2]])
cand_names = ["short", "ok", "great", "terse", "poor", "polite", "RAMBLER", "long"]
K = len(cands)
proxy_reward = cands @ w_rm              # what the reward model believes (what RL will optimise)
true_rew = true_reward(cands)            # what humans would really think (RL never sees this)

ref_logits = np.array([1.0, 0.8, 0.2, 0.9, 0.5, 0.6, -1.5, 0.1])
pi_ref = np.exp(ref_logits - ref_logits.max()); pi_ref /= pi_ref.sum()
logp_ref = np.log(pi_ref)

print(f"{'candidate':>10} | {'pi_ref':>6} | {'proxy':>6} | {'true':>7}")
for k in range(K):
    print(f"{cand_names[k]:>10} | {pi_ref[k]:6.3f} | {proxy_reward[k]:6.2f} | {true_rew[k]:7.2f}")

In [ ]:
def log_softmax(z):
    # TODO: stable log-softmax over the last axis
    pass


def kl_divergence(p, q):
    # TODO: sum over a of p_a * (log p_a - log q_a), treating 0*log(0) as 0
    pass


def rlhf_objective(pi, reward, pi_ref, beta):
    # TODO: expected reward minus beta * KL(pi || pi_ref)
    pass


def optimal_policy(reward, pi_ref, beta):
    # TODO: softmax(log(pi_ref) + reward / beta), computed stably
    pass


def reinforce_step(theta, reward, logp_ref, beta, lr, n_samples, rng):
    # TODO: one REINFORCE step as described above; return the new theta
    pass


# --- test cases: do not modify ---
z_big = np.array([1000.0, 1000.0, -1000.0])
assert np.allclose(np.exp(log_softmax(z_big)), [0.5, 0.5, 0.0])
assert np.isclose(kl_divergence(pi_ref, pi_ref), 0.0, atol=1e-12)
assert kl_divergence(np.array([0.9, 0.1]), np.array([0.5, 0.5])) > 0
assert np.isclose(kl_divergence(np.array([1.0, 0.0]), np.array([0.5, 0.5])), np.log(2))

for beta_ in [0.1, 1.0]:
    star = optimal_policy(proxy_reward, pi_ref, beta_)
    assert np.isclose(star.sum(), 1.0) and np.all(star >= 0)
    best = rlhf_objective(star, proxy_reward, pi_ref, beta_)
    rs = np.random.RandomState(0)
    for _ in range(200):                      # no random policy may beat the closed form
        other = rs.dirichlet(np.ones(K))
        assert rlhf_objective(other, proxy_reward, pi_ref, beta_) <= best + 1e-9
assert np.allclose(optimal_policy(proxy_reward, pi_ref, 1e6), pi_ref, atol=1e-4)    # huge beta -> stay at reference
assert optimal_policy(proxy_reward, pi_ref, 1e-3).argmax() == proxy_reward.argmax()  # tiny beta -> argmax reward
print("Task C2 (helper functions) checks passed.")

In [ ]:
beta_train = 0.5
theta = np.log(pi_ref).copy()            # start from the reference policy
rng_rl = np.random.RandomState(0)
objective_history = []

# TODO: for 400 steps:
#         objective_history.append(rlhf_objective(softmax(theta), proxy_reward, pi_ref, beta_train))
#         theta = reinforce_step(theta, proxy_reward, logp_ref, beta_train, lr=0.5, n_samples=2000, rng=rng_rl)
#       (softmax(theta) = np.exp(log_softmax(theta)))

pi_rl = np.exp(log_softmax(theta))
pi_star = optimal_policy(proxy_reward, pi_ref, beta_train)

# TODO: plot objective_history (x: step, y: RLHF objective) and add a horizontal line at the value of the
#       closed-form optimum rlhf_objective(pi_star, ...)

tv = 0.5 * np.abs(pi_rl - pi_star).sum()
print("pi_ref :", np.round(pi_ref, 3))
print("pi_rl  :", np.round(pi_rl, 3))
print("pi_star:", np.round(pi_star, 3))
print(f"total-variation distance(pi_rl, pi_star) = {tv:.4f}")

# --- test cases: do not modify ---
assert len(objective_history) == 400
assert objective_history[-1] > objective_history[0] + 0.1
assert tv < 0.05
print("Task C2 (REINFORCE) checks passed.")

### Task C3: Reward hacking and the role of $\beta$

Look at the supplied table above: the candidate `RAMBLER` is extremely long. The reward model has never seen
such lengths, and extrapolates its small positive length weight to give it a **high proxy reward**, while real
humans would find it awful (very negative true reward).

Using the closed-form optimal policy, sweep
`betas = [0.03, 0.1, 0.3, 1, 3, 10]`. For each $\beta$ record

- the expected **proxy** reward $\mathbb{E}_{\pi^*}[r_{proxy}]$,
- the expected **true** reward $\mathbb{E}_{\pi^*}[r_{true}]$,
- the $\mathrm{KL}(\pi^*\Vert\pi_{ref})$,

then plot proxy and true reward against $\beta$ (log x-axis) and describe what you see in Analysis Question Q4.

In [ ]:
betas = [0.03, 0.1, 0.3, 1, 3, 10]
proxy_E, true_E, kl_E = [], [], []

# TODO: for beta in betas: pi = optimal_policy(proxy_reward, pi_ref, beta);
#       append pi @ proxy_reward, pi @ true_rew, kl_divergence(pi, pi_ref)

# TODO: print a table (beta | KL | proxy reward | true reward) and plot proxy and true reward vs beta (log x)

# --- test cases: do not modify ---
assert len(proxy_E) == len(true_E) == len(kl_E) == len(betas)
assert proxy_E[0] > proxy_E[-1], "weaker KL penalty must give a higher proxy reward"
assert true_E[0] < true_E[-1], "...but here that is reward hacking: the TRUE reward gets worse"
assert kl_E[0] > kl_E[-1]
print("Task C3 checks passed.")

### Task C4 (Bonus): Direct Preference Optimisation (DPO)

RLHF needs a separate reward model *and* an RL loop. **DPO** skips both: it optimises the policy **directly on the
preference pairs**. For a chosen response $w$ and rejected response $l$:
$$
L_{DPO}(\theta) = -\frac{1}{M}\sum_m \log\sigma\!\Big(\beta\big[(\log\pi_\theta(w_m)-\log\pi_{ref}(w_m)) - (\log\pi_\theta(l_m)-\log\pi_{ref}(l_m))\big]\Big).
$$

For our softmax policy, with $z_m$ the argument of the sigmoid, the gradient simplifies to
$$\nabla_\theta L = -\frac{\beta}{M}\sum_m \sigma(-z_m)\,(\mathbf{e}_{w_m}-\mathbf{e}_{l_m}),$$
(the $\pi_\theta$ terms cancel!). Implement `dpo_loss_and_grad(theta, logp_ref, idx_w, idx_l, beta)` returning
`(loss, grad)`, verify the gradient numerically, train for 500 steps with `lr = 2.0`, `beta = 0.5`, and compare
the resulting policy's **true** expected reward with the reference policy and with the RLHF policy `pi_rl`.
Here the preference pairs are labelled by the real human-like reward (`true_rew`), *including* comparisons that
involve `RAMBLER`.

In [ ]:
ATTEMPT_BONUS = False  # set to True when you attempt the bonus task
# --- supplied: preference pairs over the 8 candidates, labelled by humans (noisy Bradley-Terry on true_rew) ---
rng_dpo = np.random.RandomState(0)
pa = rng_dpo.randint(0, K, 3000); pb = rng_dpo.randint(0, K, 3000)
keep = pa != pb
pa, pb = pa[keep], pb[keep]
a_pref = rng_dpo.rand(len(pa)) < 1 / (1 + np.exp(-(true_rew[pa] - true_rew[pb])))
idx_w = np.where(a_pref, pa, pb); idx_l = np.where(a_pref, pb, pa)


def dpo_loss_and_grad(theta, logp_ref, idx_w, idx_l, beta):
    # TODO: lp = log_softmax(theta); z = beta * ((lp[idx_w] - logp_ref[idx_w]) - (lp[idx_l] - logp_ref[idx_l]))
    #       loss = mean(np.logaddexp(0, -z));  coef = -beta * sigmoid(-z) / len(z)
    #       grad = zeros(K);  np.add.at(grad, idx_w, coef);  np.add.at(grad, idx_l, -coef)
    pass


# TODO: gradient check with numeric_grad, then train 500 steps (theta starts at log pi_ref, lr=2.0, beta=0.5)
theta_dpo = np.log(pi_ref).copy()

# TODO: print true expected reward of pi_ref, pi_rl and the DPO policy, and the probability each puts on RAMBLER

# --- test cases: do not modify ---
if ATTEMPT_BONUS:
    pi_dpo = np.exp(log_softmax(theta_dpo))
    assert pi_dpo @ true_rew > pi_ref @ true_rew
    assert pi_dpo[cand_names.index("RAMBLER")] < pi_ref[cand_names.index("RAMBLER")]
    print("Bonus Task C4 checks passed.")

# Part D: The Real Thing — `peft` and `bitsandbytes`

Everything you built in Parts A and B exists in the Hugging Face ecosystem:
[`peft`](https://huggingface.co/docs/peft) implements LoRA (and many other adapters) and
[`bitsandbytes`](https://github.com/bitsandbytes-foundation/bitsandbytes) implements 4-bit NF4 quantisation with
double quantisation. **Run this part on Google Colab** (Runtime > Change runtime type > T4 GPU). It downloads small
models (GPT-2 about 0.5 GB, OPT-125M about 0.25 GB).

> **No GPU?** `bitsandbytes` 4-bit loading needs one. In that case do Task D1 (it runs on CPU) and, instead of
> running D2, write 5-6 sentences in the markdown cell under D2 describing what each argument of
> `BitsAndBytesConfig` does and what you would expect `get_memory_footprint()` to show. That written
> answer replaces the D2 code.

### Task D1: LoRA on GPT-2 with `peft`

1. Create a `LoraConfig` with `r=8`, `lora_alpha=16`, `lora_dropout=0.05`, `target_modules=["c_attn"]` (GPT-2's
   fused query/key/value projection), `fan_in_fan_out=True` (GPT-2 stores weights transposed) and
   `task_type="CAUSAL_LM"`, then wrap the model with `get_peft_model`.
2. Call `print_trainable_parameters()`.
3. **Check it against your own formula from Task A1.** GPT-2 small has 12 transformer blocks, and each `c_attn` is
   a $768 \to 2304$ layer. Compute the expected number of trainable parameters with `count_params`-style
   arithmetic and assert that it equals what `peft` reports.

In [ ]:
# [COLAB-ONLY] Task D1
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "peft", "accelerate", "bitsandbytes"])

import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

base_model = AutoModelForCausalLM.from_pretrained("gpt2")
n_total = sum(p.numel() for p in base_model.parameters())

# TODO: lora_cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=["c_attn"],
#                             fan_in_fan_out=True, task_type="CAUSAL_LM")
# TODO: peft_model = get_peft_model(base_model, lora_cfg); peft_model.print_trainable_parameters()

# TODO: n_trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
# TODO: expected = <your formula: 12 blocks * r * (d_in + d_out)>
# assert n_trainable == expected
# print(f"trainable: {n_trainable:,} of {n_total:,} ({100 * n_trainable / n_total:.3f}%)")

### Task D2: QLoRA with `bitsandbytes`

1. Build a `BitsAndBytesConfig` with `load_in_4bit=True`, `bnb_4bit_quant_type="nf4"`,
   `bnb_4bit_use_double_quant=True` and `bnb_4bit_compute_dtype=torch.float16`.
2. Load `facebook/opt-125m` twice: once in fp16 and once with the 4-bit config
   (`quantization_config=..., device_map="auto"`). Print `get_memory_footprint()` of both in MB and the ratio.
3. Prepare the 4-bit model with `prepare_model_for_kbit_training`, attach LoRA
   (`r=8`, `lora_alpha=16`, `target_modules=["q_proj", "v_proj"]`) and print the trainable parameters.
4. In the markdown cell below the code, answer: **why is the memory ratio smaller than 4x?** (Hint: which
   modules of the model are *not* quantised?)

In [ ]:
# [COLAB-ONLY] Task D2 (needs a GPU runtime)
from transformers import BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

# TODO: bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
#                                    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
# TODO: model_fp16 = AutoModelForCausalLM.from_pretrained("facebook/opt-125m", torch_dtype=torch.float16)
# TODO: model_4bit = AutoModelForCausalLM.from_pretrained("facebook/opt-125m", quantization_config=bnb_cfg, device_map="auto")
# TODO: print memory footprints (MB) and their ratio:  model.get_memory_footprint() / 1e6
# TODO: model_4bit = prepare_model_for_kbit_training(model_4bit)
# TODO: qlora_model = get_peft_model(model_4bit, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
#                                    target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"))
# TODO: qlora_model.print_trainable_parameters()

**Your answer to D2 step 4 (fill in):**

## Analysis Questions (write your answers in this cell)

**Q1. Rank and scale (Part A).** Using your rank sweep, explain why `r = 1` failed but `r = 2` succeeded, and why
`r = 16` did not help much more. In a real model, how would you choose `r`? What does `alpha / r` do, and what would
happen if you doubled `alpha` while keeping everything else fixed?

**Q2. Zero initialisation (Part A).** Why is `B` initialised to zero but `A` is not? What would go wrong if *both*
were initialised to zero? (Hint: look at the formulas for `dA` and `dB` at the first step.)

**Q3. What QLoRA saves (Part B).** In QLoRA, which tensors are stored in 4 bits, which stay in higher precision, and
through which tensors do gradients flow? Use your Task B3 numbers to say how much of the quantisation error the
adapter recovered, and explain why NF4 beat int4 on the same bell-shaped weights.

**Q4. Reward hacking (Part C).** Describe the curves in your Task C3 plot. At small `beta` the proxy reward is
high but the true reward is terrible; why? How does a larger `beta` protect against this, and what is the price
you pay for it? Name one practical way, other than a KL penalty, to reduce reward hacking.

**Q5. RLHF vs DPO (Part C).** List two things RLHF needs that DPO does not. Name one situation where you would
still prefer the RLHF pipeline (a separate reward model) over DPO.